# all-reduce-eval-metrics — worked example 3: Compute global eval RMSE from packed (sse, count) all_reduce

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-eval-metrics`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Root-mean-squared-error over a sharded test set cannot be averaged from per-rank RMSEs — the square root is nonlinear. The reducible quantities are the *sum of squared errors* and the *sample count*. Pack `(sse, count)` into one length-2 tensor, `all_reduce(SUM)`, then take `sqrt(global_sse / global_count)` once on every rank.

## Worked solution

**Step 1 — build per-rank squared-error sums.** We draw a fixed set of `(pred, target)` pairs and split them across `world_size=3` ranks. Each rank computes `sse = sum((pred - target)**2)` and its own `count`. RMSE itself is *not* summable, so we never reduce RMSE directly.

**Step 2 — pack the two reducible scalars.** `stats = t.tensor([sse, float(count)])`. Float dtype because `all_reduce` wants floats and because `sse` is already float; the count is cast for uniformity.

**Step 3 — one SUM all_reduce.** After `all_reduce(stats, SUM)`, `stats[0]` is the global sum of squared errors and `stats[1]` is the total number of samples, identical on each rank.

**Step 4 — finalize with a single sqrt-of-mean.** `rmse = sqrt(stats[0] / stats[1])`. Doing the division and square root *after* the reduce is exactly what makes the result equal to RMSE computed on the full dataset in one process — which we check against.

**Step 5 — confirm.** Because the mock sums the rank tensors, `global_sse` equals the SSE over all errors concatenated, so `rmse` matches the single-process reference RMSE to floating-point precision.

In [ ]:
import math

class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def global_rmse(dist_module, local_sse, local_count):
    stats = t.tensor([float(local_sse), float(local_count)], dtype=t.float32)
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    return math.sqrt((stats[0] / stats[1]).item())

t.manual_seed(0)
preds = t.randn(30)
targets = t.randn(30)
err = (preds - targets) ** 2
shards = err.chunk(3)
rank_tensors = [t.tensor([s.sum().item(), float(s.numel())], dtype=t.float32) for s in shards]
mock = MockDist(rank_tensors)
rmse = global_rmse(mock, shards[0].sum().item(), shards[0].numel())
ref = math.sqrt(err.mean().item())
print('global rmse:', round(rmse, 6))
print('single-process reference:', round(ref, 6))